In [9]:
import os

print("Deep scanning Kaggle's internal input directory...\n")

found_path = None
# We use os.walk to go as deep as Kaggle buried the dataset
for root, dirs, files in os.walk('/kaggle/input'):
    if 'model1_train_kag.csv' in files:
        found_path = root
        break

if found_path is None:
    print("❌ CRITICAL ERROR: Could not find 'model1_train_kag.csv' anywhere!")
    print("HOW TO FIX: Make sure the dataset is actually attached to this notebook via the right sidebar (+ Add Data).")
else:
    DATASET_DIR = found_path
    print(f"✅ Success! The real, verified path is: {DATASET_DIR}\n")
    
    print("Files in dataset root:")
    for f in os.listdir(DATASET_DIR)[:10]:
        print(f"  - {f}")

    csv_path = os.path.join(DATASET_DIR, 'model1_train_kag.csv')
    print(f"\nCSV exists: {os.path.exists(csv_path)}")

    img_path = os.path.join(DATASET_DIR, 'model1_images')
    print(f"Images folder exists: {os.path.exists(img_path)}")
    
    # Quick count of images just to be absolutely sure
    if os.path.exists(img_path):
        img_folders = os.listdir(img_path)
        print(f"Found image categories: {img_folders}")


Deep scanning Kaggle's internal input directory...

✅ Success! The real, verified path is: /kaggle/input/datasets/muhammadshabih/model1

Files in dataset root:
  - model1_test_kag.csv
  - model1_val_kag.csv
  - model1_train_kag.csv
  - model1_images

CSV exists: True
Images folder exists: True
Found image categories: ['dress', 'jumpsuit', 'outwear', 'top', 'bottom', 'shoes']


In [ ]:
# Model 1 Kaggle Training Script (Peer-Reviewed Architecture)
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

# ─────────────────────────────────────────────
# 1. CONFIGURATION
# ─────────────────────────────────────────────
# UPDATE THIS: Check your Kaggle dataset name in the input panel
DATASET_DIR = '/kaggle/input/datasets/muhammadshabih/model1'
BATCH_SIZE  = 64       # reduced from 128 — safer for GPU memory
IMG_SIZE    = (256, 256)
EPOCHS_S1   = 5        # Stage 1: head only
EPOCHS_S2   = 15       # Stage 2: fine-tuning
NUM_CLASSES = 6
CATEGORIES  = ['top', 'bottom', 'outwear', 'shoes', 'dress', 'jumpsuit']

cat2idx = {cat: idx for idx, cat in enumerate(CATEGORIES)}
idx2cat = {idx: cat for cat, idx in cat2idx.items()}

print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))

# ─────────────────────────────────────────────
# 2. LOAD DATA
# ─────────────────────────────────────────────
train_df = pd.read_csv(os.path.join(DATASET_DIR, 'model1_train_kag.csv'))
val_df   = pd.read_csv(os.path.join(DATASET_DIR, 'model1_val_kag.csv'))

train_df['image_path'] = DATASET_DIR + '/' + train_df['image_path']
val_df['image_path']   = DATASET_DIR + '/' + val_df['image_path']

train_df['label'] = train_df['category'].map(cat2idx)
val_df['label']   = val_df['category'].map(cat2idx)

# Verify paths exist
train_df = train_df[train_df['image_path'].apply(os.path.exists)].reset_index(drop=True)
val_df   = val_df[val_df['image_path'].apply(os.path.exists)].reset_index(drop=True)

print(f"Train size: {len(train_df):,}")
print(f"Val size:   {len(val_df):,}")
print(f"\nCategory distribution (train):")
print(train_df['category'].value_counts())

# ─────────────────────────────────────────────
# 3. CLASS WEIGHTS — critical for imbalance
# ─────────────────────────────────────────────
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_df['label'].values
)
class_weights = {i: w for i, w in enumerate(class_weights_array)}
print(f"\nClass weights (handles imbalance):")
for idx, weight in class_weights.items():
    print(f"  {idx2cat[idx]:<12} weight: {weight:.3f}")

# ─────────────────────────────────────────────
# 4. DATA PIPELINE
# ─────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

def parse_image(filename, label):
    img = tf.io.read_file(filename)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.cast(img, tf.float32)
    return img, label

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.15)
    image = tf.image.random_contrast(image, lower=0.85, upper=1.15)
    image = tf.image.random_saturation(image, lower=0.85, upper=1.15)
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, label

train_ds = (
    tf.data.Dataset.from_tensor_slices(
        (train_df['image_path'].values, train_df['label'].values)
    )
    .shuffle(buffer_size=20000, seed=42)
    .map(parse_image, num_parallel_calls=AUTOTUNE)
    .map(augment, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices(
        (val_df['image_path'].values, val_df['label'].values)
    )
    .map(parse_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# ─────────────────────────────────────────────
# 5. BUILD MODEL — named embedding layer
# ─────────────────────────────────────────────
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(256, 256, 3)
)

# Start with base fully frozen
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D(name='embedding_layer')(x)  # named for reliable extraction
x = Dropout(0.4)(x)
predictions = Dense(NUM_CLASSES, activation='softmax', name='classifier')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# ─────────────────────────────────────────────
# 6. STAGE 1 — Train head only (base frozen)
# ─────────────────────────────────────────────
print("\n" + "="*50)
print("STAGE 1 — Training classification head only")
print("Base model: FROZEN")
print("="*50)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Trainable params: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")

history_s1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_S1,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    ]
)

# ─────────────────────────────────────────────
# 7. STAGE 2 — Fine-tune top layers (lower lr)
# ─────────────────────────────────────────────
print("\n" + "="*50)
print("STAGE 2 — Fine-tuning top 20 layers")
print("Base model: top 20 layers UNFROZEN")
print("="*50)

# Unfreeze top 20 layers only
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Must recompile after changing trainable state
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Trainable params: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")

callbacks_s2 = [
    ModelCheckpoint(
        'model1_efficientnet_best.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

history_s2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_S2,
    class_weight=class_weights,
    callbacks=callbacks_s2
)

# ─────────────────────────────────────────────
# 8. EVALUATION — per class accuracy
# ─────────────────────────────────────────────
print("\n" + "="*50)
print("EVALUATION — Per-class accuracy")
print("="*50)

y_true = val_df['label'].values
y_pred_probs = model.predict(val_ds, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

from sklearn.metrics import classification_report, confusion_matrix
report = classification_report(
    y_true[:len(y_pred)],
    y_pred,
    target_names=CATEGORIES
)
print(report)

# Save evaluation report
with open('model1_evaluation.txt', 'w') as f:
    f.write("Model 1 — EfficientNetB0 Clothing Classifier\n")
    f.write("="*50 + "\n")
    f.write(report)

print("Evaluation saved to model1_evaluation.txt")

# ─────────────────────────────────────────────
# 9. SAVE EMBEDDING EXTRACTOR
# ─────────────────────────────────────────────
# Extract from the named GlobalAveragePooling2D layer
# This gives 1280-dimensional vectors for Model 2
embedding_model = Model(
    inputs=model.input,
    outputs=model.get_layer('embedding_layer').output
)
embedding_model.save('model1_embedding_extractor.h5')

print("\n" + "="*50)
print("TRAINING COMPLETE — Download these files:")
print("="*50)
print("1. model1_efficientnet_best.h5")
print("   → Classifies uploaded clothing into 6 categories")
print("2. model1_embedding_extractor.h5")
print("   → Extracts 1280-dim vectors for Model 2 training")
print("3. model1_evaluation.txt")
print("   → Per-class accuracy for FYP report")
print("="*50)

TensorFlow version: 2.19.0
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Train size: 248,619
Val size:   53,536

Category distribution (train):
category
bottom      121731
top          66654
dress        32261
outwear      21042
shoes         5744
jumpsuit      1187
Name: count, dtype: int64

Class weights (handles imbalance):
  top          weight: 0.622
  bottom       weight: 0.340
  outwear      weight: 1.969
  shoes        weight: 7.214
  dress        weight: 1.284
  jumpsuit     weight: 34.909


I0000 00:00:1774109450.829306      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774109450.835274      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step

STAGE 1 — Training classification head only
Base model: FROZEN
Trainable params: 7,686
Epoch 1/5


I0000 00:00:1774109465.021642     150 service.cc:152] XLA service 0x7a9d74211ba0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774109465.021683     150 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1774109465.021688     150 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1774109467.466740     150 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-03-21 16:11:16.278983: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-21 16:11:16.442990: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-21 16:11:16.863152: E external/local_xl

3884/3885 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - accuracy: 0.7181 - loss: 0.5923

...

3885/3885 ━━━━━━━━━━━━━━━━━━━━ 433s 111ms/step - accuracy: 0.8738 - loss: 0.2135 - val_accuracy: 0.8780 - val_loss: 0.3071 - learning_rate: 1.0000e-04
Epoch 5/15
3885/3885 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.8807 - loss: 0.1977
Epoch 5: val_accuracy improved from 0.87799 to 0.88613, saving model to model1_efficientnet_best.h5


3885/3885 ━━━━━━━━━━━━━━━━━━━━ 433s 112ms/step - accuracy: 0.8807 - loss: 0.1977 - val_accuracy: 0.8861 - val_loss: 0.2840 - learning_rate: 1.0000e-04
Epoch 6/15
3885/3885 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.8866 - loss: 0.1790
Epoch 6: val_accuracy did not improve from 0.88613
3885/3885 ━━━━━━━━━━━━━━━━━━━━ 432s 111ms/step - accuracy: 0.8866 - loss: 0.1790 - val_accuracy: 0.8818 - val_loss: 0.2939 - learning_rate: 1.0000e-04
Epoch 7/15
3885/3885 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.8926 - loss: 0.1687
Epoch 7: val_accuracy improved from 0.88613 to 0.88701, saving model to model1_efficientnet_best.h5


3885/3885 ━━━━━━━━━━━━━━━━━━━━ 434s 112ms/step - accuracy: 0.8926 - loss: 0.1687 - val_accuracy: 0.8870 - val_loss: 0.2769 - learning_rate: 1.0000e-04
Epoch 8/15
3885/3885 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.8959 - loss: 0.1611
Epoch 8: val_accuracy improved from 0.88701 to 0.89099, saving model to model1_efficientnet_best.h5


3885/3885 ━━━━━━━━━━━━━━━━━━━━ 433s 112ms/step - accuracy: 0.8959 - loss: 0.1611 - val_accuracy: 0.8910 - val_loss: 0.2659 - learning_rate: 1.0000e-04
Epoch 9/15
3885/3885 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.8997 - loss: 0.1496
Epoch 9: val_accuracy did not improve from 0.89099
3885/3885 ━━━━━━━━━━━━━━━━━━━━ 434s 112ms/step - accuracy: 0.8997 - loss: 0.1496 - val_accuracy: 0.8901 - val_loss: 0.2685 - learning_rate: 1.0000e-04
Epoch 10/15
3885/3885 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.9030 - loss: 0.1429
Epoch 10: val_accuracy improved from 0.89099 to 0.89159, saving model to model1_efficientnet_best.h5


3885/3885 ━━━━━━━━━━━━━━━━━━━━ 433s 111ms/step - accuracy: 0.9030 - loss: 0.1429 - val_accuracy: 0.8916 - val_loss: 0.2579 - learning_rate: 1.0000e-04
Epoch 11/15
3885/3885 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.9063 - loss: 0.1333
Epoch 11: val_accuracy did not improve from 0.89159
3885/3885 ━━━━━━━━━━━━━━━━━━━━ 434s 112ms/step - accuracy: 0.9063 - loss: 0.1333 - val_accuracy: 0.8912 - val_loss: 0.2628 - learning_rate: 1.0000e-04
Epoch 12/15
3885/3885 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.9089 - loss: 0.1277
Epoch 12: val_accuracy did not improve from 0.89159

Epoch 12: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
3885/3885 ━━━━━━━━━━━━━━━━━━━━ 432s 111ms/step - accuracy: 0.9089 - loss: 0.1277 - val_accuracy: 0.8904 - val_loss: 0.2600 - learning_rate: 1.0000e-04
Epoch 13/15
 894/3885 ━━━━━━━━━━━━━━━━━━━━ 4:42 94ms/step - accuracy: 0.9087 - loss: 0.1275